In [4]:
import os

os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'  # MUST be before unsloth import

os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

print("ModelScope:", os.environ['UNSLOTH_USE_MODELSCOPE'])
print("LD set ✓")

ModelScope: 1
LD set ✓


In [2]:
!pip install modelscope -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 85.5 MB/s eta 0:00:00


In [5]:
# Install unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl==0.14.0 peft==0.14.0 -q
print("✅ Unsloth installed!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 9.6 MB/s eta

In [6]:
 #Step3: Import necessary libraries
import unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from trl import SFTTrainer
from huggingface_hub import login
from transformers import TrainingArguments
from datasets import load_dataset
import wandb
print("✅ Imports done")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
✅ Imports done


In [7]:
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [8]:
# Optional: Checking GPU availability
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "No GPU"


CUDA available: True
GPU device: Tesla T4


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",  # unsloth/ prefix for ModelScope
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    token = hf_token,
)
print("✅ Model loaded!")

2026-06-12 17:53:29,914 - modelscope - INFO - Got 9 files, start to download ...


Processing 9 items:   0%|          | 0.00/9.00 [00:00<?, ?it/s]

In [ ]:
#Setup system prompt
prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Below crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response in both and accurate.

### Instruction:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}

### Response:
{}
"""

In [ ]:
# Run Inference on the model

# Define a test question
question = """ A 61-year-old woman with a long history of involuntary urine loss duing activities like coughing or sneezing but no leakage at night undergoes a gynecological eaxm and Q-tip test. Based on these findings,what would cystometry most likely reveal about her residual volume and detrusor contractions.
"""

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200, # Corrected from max_new_token to max_new_tokens
    use_cache = True
)

# Decode the response tokens back to text
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)

clean = response[0]

clean = clean.replace("Ġ", " ")
clean = clean.replace("Ċ", "\n")

# Extract only the answer after <think>...</think>
if "</think>" in clean:
    clean = clean.split("</think>")[-1].strip()
elif "### Response:" in clean:
    clean = clean.split("### Response:")[-1].strip()

print(clean)


In [ ]:
# setup fine-tuning

# Load Dataset
medical_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT","en",split = "train[:500]", trust_remote_code = True)

In [ ]:
medical_dataset[1]

In [ ]:
EOS_TOKEN = tokenizer.eos_token
EOS_TOKEN
# Define EOS_TOKEN (End Of Sentence) which tells the model when to stop generating text during training

In [ ]:
# Update the prompt_style for training

train_prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Below crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response in both and accurate.

### Instruction:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

In [ ]:

# Prepare the data for fine-tuning

def preprocess_input_data(examples):
  inputs = examples["Question"]
  cots = examples["Complex_CoT"]
  outputs = examples["Response"]

  texts = []
  for input, cot, output in zip(inputs, cots, outputs):
    text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
    texts.append(text)

  return {
      "texts" : texts,
  }

In [ ]:
finetune_dataset = medical_dataset.map(
    preprocess_input_data,
    batched=True
)

In [ ]:
finetune_dataset["texts"][1]

In [ ]:
# Setup/Apply LoRA finetuning to the model

model_lora = FastLanguageModel.get_peft_model(
    model = model,
    r = 16, # New adepter size 16 which is going to trained
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    use_rslora = False,
    loftq_config = None
)

In [ ]:
# Add this before creating the trainer
if hasattr(model, '_unwrapped_old_generate'):
  del model._unwrapped_old_generate

In [ ]:
print(type(finetune_dataset["texts"][0]))
print(finetune_dataset["texts"][0][:500])

In [ ]:
print(finetune_dataset["texts"][0][-1000:])

In [ ]:
# Define the formatting function as required by Unsloth's SFTTrainer
max_sequence_length = 2048

def formatting_prompts_func(examples):
    # `examples["texts"]` will be a list of lists, where each inner list contains one formatted string.
    # We need to flatten this into a single list of strings.
    flattened_texts = [text for sublist in examples["texts"] for text in sublist]
    return flattened_texts

trainer = SFTTrainer(
    model = model_lora,
    tokenizer = tokenizer,
    train_dataset = finetune_dataset,
    # The 'dataset_text_field' is not used when 'formatting_func' is provided.
    # It seems Unsloth's SFTTrainer explicitly requires 'formatting_func' in this context.
    formatting_func = formatting_prompts_func, # Pass the formatting function
    max_seq_length = max_sequence_length,
    dataset_num_proc = 1,

    # Define training args
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir = "outputs",
        gradient_checkpointing = False # Explicitly set gradient_checkpointing to False
        save_strategy = "no",
        report_to= "none",
    )
)

In [ ]:
# Setup, WANDB
from google.colab import userdata
wnb_token = userdata.get("WANDB_API_TOKEN")
#login to WnB
wandb.login(key=wnb_token) #import wandb
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-on-Medical-CoT-Dataset',
    job_type="training",
    anonymous="allow"
)


In [ ]:
# Fix PicklingError - disable checkpoint saving
import os

# Training already completed! Just save the final model directly
model_lora.save_pretrained("outputs/final_model")
tokenizer.save_pretrained("outputs/final_model")
print("✅ Model saved successfully!")

In [ ]:
import torch

# Start the fine-tuning process
trainer_stats = trainer.train()

In [ ]:
wandb.finish()

In [ ]:
# Testing after finetuning
question = """ A 61-year-old woman with a long history of involuntary urine loss duing activities like coughing or sneezing but no leakage at night undergoes a gynecological eaxm and Q-tip test. Based on these findings,what would cystometry most likely reveal about her residual volume and detrusor contractions.
"""
# Ensure FastLanguageModel is defined in this scope
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model_lora)

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model_lora.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200, # Corrected from max_new_token to max_new_tokens
    use_cache = True
)

# Decode the response tokens back to text
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = response.replace("Ġ", " ").replace("Ċ", "\n")
if "</think>" in response:
    response = response.split("</think>")[-1].strip()

print(response)